In [11]:
import time
from IPython.display import display
import kbio.kbio_run_techniques as techs
import kbio.kbio_run
from kbio.kbio_tech import get_info_data
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Connect to the potentiostat and prepare the instrument for running techniques.

The `connect_potentiostat()` helper from `kbio.kbio_run`:
- loads the EcLab DLL from `ECLIB_DIR` or the default path
- connects to the potentiostat at the specified `address` and `channel`
- detects the board type and prints the recommended `.ecc` technique file set
- optionally loads firmware when `load_firmware=True`
- checks that the channel kernel is loaded before returning the API object

In [ ]:
address = "192.168.0.91"
channel = 1
# Connect to the potentiostat. This function loads the EcLab DLL, connects to the
# specified instrument, detects the board type, prints which .ecc files should be
# used for the selected technique family, and optionally loads firmware.
api, id_ = kbio.kbio_run.connect_potentiostat(address, channel, load_firmware=True)
board_type = api.GetChannelBoardType(id_, channel)

In [3]:
# EIS measurement setup parameters
# - voltage: target cell voltage for the experiment
# - voltage_time: hold time at the set voltage for both CA and PEIS
# - ocv_time: open-circuit hold time used between CA and PEIS
voltage = 1.46
voltage_time = 30
ocv_time = 60

# Frequency sweep settings for PEIS
points_per_decade = 6
f_start = 20000
f_end = 2
f_list = np.logspace(np.log10(f_start), np.log10(f_end), np.ceil(points_per_decade * int(abs(np.log10(f_start / f_end)))))
# Randomize the order of frequencies to avoid systematic sweep ordering effects
randomize = False
if randomize:
    np.random.shuffle(f_list)
print(f_list)

# Maximum number of initial OCV and CA cycles before current is stabilized
repeats = 20

# Set how much close two subsequent currents must be to each other to be considered as stable in percent.
# E.g. 0.2 means that 0.998 < average(I_(n+1)/I_n) < 1.002
stabilization_range = 0.1

# Create the technique parameter objects used by the instrument API
ca_params = techs.CA_params(
    api,
    Voltage_step=voltage,
    vs_init=False,
    Duration_step=voltage_time,
    Record_every_dT=0.1,
    Record_every_dI=1,
    N_cycles=0,
    I_range=11,
    E_range=3,
)
# OCV technique parameters define how long the open-circuit hold lasts
ocv_params = techs.OCV_params(api, Rest_time_T=ocv_time, Record_every_dT=0.1, Record_every_dE=1)

# LOOP parameters determine the protocol sequencing for the technique chain
loop_params = techs.LOOP_params(api, loop_N_times=-1, protocol_number=1)
loop_params_init = techs.LOOP_params(api, loop_N_times=-1, protocol_number=0)


[2.00000000e+04 1.34003750e+04 8.97850252e+03 6.01576504e+03
 4.03067537e+03 2.70062808e+03 1.80947145e+03 1.21237980e+03
 8.12317198e+02 5.44267754e+02 3.64669600e+02 2.44335470e+02
 1.63709346e+02 1.09688332e+02 7.34932388e+01 4.92418480e+01
 3.29929615e+01 2.21059028e+01 1.48113694e+01 9.92389521e+00
 6.64919586e+00 4.45508590e+00 2.98499109e+00 2.00000000e+00]


NameError: name 'api' is not defined

In [13]:
voltage_str = str(voltage).replace('.', ',')  # convert voltage to string and replace decimal point with comma
ocv_time_str = str(ocv_time).replace('.', ',')  # convert ocv_time to string and replace decimal point with comma
voltage_time_str = str(voltage_time).replace('.', ',')  # convert voltage_time to string and replace decimal point with comma

# Path for saving output files
output_folder = r'\\ELECTROLYZER\PEM-WE_measurements\2026\414_VIII_VIII_IrOx_5minPt_18nm_N212_plain_BDC903Ptwireasreference'  
output_file = "trpeis_at_" + voltage_str + "V_for_" + voltage_time_str + "s_and_" + ocv_time_str + "s_OCV_"  # build base filename with experiment parameters
output_file_init = output_file + "_init"  # create a filename for data from initial stabilization cycles

  # list existing files that match the output file base name, so you can adjust the filename so new files are created for the new measurement
kbio.kbio_run.print_matching_files(output_folder, output_file)


No files found in \\ELECTROLYZER\PEM-WE_measurements\2026\414_VIII_VIII_IrOx_5minPt_18nm_N212_plain_BDC903Ptwireasreference containing 'trpeis_at_1,46V_for_30s_and_60s_OCV_0_'


In [ ]:
"""
Initial loops for system stabilization
"""

techniqs = ['CA','OCV']  # techniques to include for the initial stabilization phase

index = 0  # loop index for the current data segment
data_out_init = kbio.kbio_run.create_data_out(techniqs)  # initialize the data structure for initial loops
data_out_init_all = []  # store all initial loop data sets for stability checking

api.LoadTechnique(id_, channel, 'ocv.ecc', ocv_params, first=True, last=False)  # load OCV technique first
api.LoadTechnique(id_, channel, 'ca.ecc', ca_params, first=False, last=False)  # load CA technique next
api.LoadTechnique(id_, channel, 'loop.ecc', loop_params_init, first=False, last=True)  # load loop controller for the init sequence

api.StartChannel(id_, channel)  # start the instrument channel for initial loops

while True:
    data = api.GetData(id_, channel)  # fetch the latest data package from the instrument

    if len(data[2]) > 0:  # proceed only when there is actual recorded data
        status, tech_name = get_info_data(api, data)  # decode technique and status
        data_out_init = kbio.kbio_run.get_exp_data(api, data, board_type=board_type, data_out=data_out_init, index=index)  # parse and append data

        print('\rInitial loops running at ' + str(tech_name) + ' for ' + str(round(data[0].ElapsedTime, 4)) + ' s', end='', flush=True)  # live status update

        time.sleep(1)  # throttle updates during the initial loops

        if index < data[1].loop:  # only save loop data when the loop index is valid
            data_out_init_all.append(data_out_init)  # keep a copy of the current initial loop data

            if index > 3:  # At least three stabilization cycles has to run
                mask1 = np.array(data_out_init_all[index]['CA']['time/s']) > 1  # ignore first second of CA data
                mask2 = np.array(data_out_init_all[index-1]['CA']['time/s']) > 1  # ignore first second of previous CA cycle

                # compare current stabilization between consecutive CA cycles
                avg = np.mean(np.array(data_out_init_all[index]['CA']['I/mA'])[mask1] / np.array(data_out_init_all[index-1]['CA']['I/mA'])[mask2])
                print(avg)
                if 1-stabilization_range/100 < avg < 1+stabilization_range/100:
                    print(f'Current stabilized in cycle {index}')
                    break

            index = data[1].loop  # update index from the instrument loop counter

            with plot_output:
                plot_output.clear_output(wait=True)  # clear previous plot output
                fig = kbio.kbio_run.plot_results(data_out_init, techniqs, index)  # generate live plot
                display(fig)
                plt.close(fig)

            save_data = kbio.kbio_run.save_data(data_out_init, output_folder, output_file_init)  # save current init data to disk
            data_out_init = kbio.kbio_run.create_data_out(techniqs)  # reset the init data buffer for the next loop

        if index > repeats:  # safety break to avoid indefinite initial looping
            break

api.StopChannel(id_, channel)    
print('Initial loops finished. ')

"""
Impedance spectroscopy measurement at different frequencies
"""
techniqs = ['OCV','CA','PEIS']
# list to accumulate results from each frequency
data_out_all = []

for index, f in enumerate(f_list):
    data_out = kbio.kbio_run.create_data_out(techniqs)  # create an empty data container for this frequency


    # Load pre-conditioning techniques: OCV then CA
    api.LoadTechnique(id_, channel, 'ocv.ecc', ocv_params, first=True, last=False)  # hold voltage before conditioning
    api.LoadTechnique(id_, channel, 'ca.ecc', ca_params, first=False, last=True)  # current application for stabilization

    api.StartChannel(id_, channel)  # start the channel to execute the queued techniques

    print(f"Reading data for {f} Hz")  # announce the frequency being measured

    while True:
        data = api.GetData(id_, channel)  # poll the instrument for new data

        if len(data[2]) > 0:
            status, tech_name = get_info_data(api, data)  # decode technique name and status from the packet
            data_out = kbio.kbio_run.get_exp_data(api, data, board_type=board_type, data_out=data_out, index=index)  # parse and append data

            # live progress output; carriage return keeps the print on one line
            print('\rFrequency ' + str(f) + ' Hz running at ' + str(tech_name) + ' for ' + str(round(data[0].ElapsedTime, 4)) + ' s', end="", flush=True)
            if tech_name == 'OCV':
                time.sleep(1)  # pause briefly during OCV updates to reduce CPU load
            elif tech_name == 'CA':
                time.sleep(1)  # pause briefly during CA updates

            if status == "STOP":
                break  # pre-conditioning finished, exit loop
    api.StopChannel(id_, channel)  # stop the channel after pre-conditioning


    # Prepare and run PEIS at the current frequency
    api.LoadTechnique(id_, channel, 'ocv.ecc', ocv_params, first=True, last=False)  # ensure voltage hold before PEIS
    peis_params = techs.PEIS_params(
        api,
        vs_init=False,
        Initial_Voltage_step=voltage,
        step_duration=0,
        record_dt=1,
        record_dE=1,
        Final_frequency=f,
        Initial_frequency=f,
        Lin_Log=True,
        Amplitude=0.005,
        Frequency_number=1,
        Average_N_times=1,
        Correction=False,
        Wait_for_steady=0.1,
        I_range=11,
    )  # configure PEIS parameters specific to this frequency
    api.LoadTechnique(id_, channel, 'peis.ecc', peis_params, first=False, last=False)  # load PEIS technique
    api.LoadTechnique(id_, channel, 'loop.ecc', loop_params, first=False, last=True)  # load loop controller to finish sequence

    # Start PEIS sequence
    api.StartChannel(id_, channel)

    while True:
        data = api.GetData(id_, channel)  # read data while PEIS runs

        if len(data[2]) > 0:
            status, tech_name = get_info_data(api, data)  # get current technique/status
            # extend elapsed time to account for hold durations
            if data[0].ElapsedTime < (voltage_time + ocv_time):
                data[0].ElapsedTime += (voltage_time + ocv_time)  # adjust elapsed time for display
                data_out = kbio.kbio_run.get_exp_data(api, data, board_type=board_type, data_out=data_out, index=index)  # append PEIS data

                # live PEIS progress update
                print('\rFrequency ' + str(f) + ' Hz running at ' + str(tech_name) + ' for ' + str(round(data[0].ElapsedTime, 4)) + ' s', end="", flush=True)
                if tech_name == 'OCV':
                    time.sleep(1)  # throttle updates during OCV
                else:
                    pass  # no delay for other states
            else:
                print(" ")
                print(f'Measurement at {f} Hz finished.')  # end-of-frequency message
                break
    api.StopChannel(id_, channel)  # stop channel when PEIS completes

    data_out_all.append(data_out)  # collect this frequency's data

    with plot_output:
        plot_output.clear_output(wait=True)  # clear prior plot
        fig = kbio.kbio_run.plot_results(data_out_all, techniqs, index)  # plot accumulated results so far
        display(fig)
        plt.close(fig)

    save_data = kbio.kbio_run.save_data(data_out, output_folder, output_file)  # save the current data files
    index += 1  # increment measurement counter
api.Disconnect(id_)  # disconnect from instrument at end of sweep

warning.close()  # close the 'experiment running' widget
warning = kbio.kbio_run.experiment_warning(address, running=False)  # create a widget to indicate completion
print("Experiment completed")

